# Gold layer —  metadata / extraction

Pipeline before model use:
upload pdf -> extract to txt -> clean txt -> preprocess for nlp -> metadata extraction -> classification -> output metadata


In [68]:
import json
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional
from collections import Counter

import pandas as pd
import torch
from transformers import pipeline

In [69]:
SILVER_FOLDER = Path("../../../Data/silver")
GOLD_FOLDER = Path("../../../Data/gold/MBART/metadata")
REPORTS_FOLDER = GOLD_FOLDER / "reports"

GOLD_FOLDER.mkdir(parents=True, exist_ok=True)
REPORTS_FOLDER.mkdir(parents=True, exist_ok=True)

In [70]:
DOC_ID_COLUMN = "document_id"
RAW_TEXT_COLUMN = "raw_text"
NLP_TEXT_COLUMN = "nlp_text"
TEXT_COLUMN = "cleaned_text"
ENTITIES_COLUMN = "entities"

In [71]:
ZERO_SHOT_MODEL = "facebook/bart-large-mnli"

RESEARCH_GROUPS = [
    "Aquaculture in Delta Areas",
    "Assetmanagement",
    "Biobased Bouwen",
    "Building with Nature",
    "Data Science",
    "Delta Power",
    "Excellence and Innovation in Education",
    "HZ Kenniscentrum Kusttoerisme",
    "HZ Kenniscentrum Ondernemen en Innoiveren",
    "HZ Kenniscentrum Zeeuwse Samenleving",
    "HZ Nexus",
    "Healthy Region",
    "Kunst Cultuur Transitie",
    "Marine Biobased Chemie",
    "Ouderenzorg",
    "Resilient Deltas",
    "Supply Chain Innovation",
    "Water Technology",
]

RESEARCH_GROUP_CONFIDENCE_THRESHOLD = 0.20

zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model=ZERO_SHOT_MODEL,
    device=0 if torch.cuda.is_available() else -1
)

print("Loaded zero-shot classifier.")

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 12733.09it/s]


Loaded zero-shot classifier.


In [72]:
STRUCTURAL_PREFIXES = {
    "collected proof",
    "appendix",
    "appendixes",
    "figure",
    "table of contents",
    "contents",
    "subject",
    "keywords",
    "author",
    "authors",
    "references",
    "bibliography",
}

SECTION_LABELS = {
    "abstract", "introduction", "background", "discussion", "conclusion",
    "conclusions", "results", "result", "method", "methods", "methodology",
    "references", "bibliography", "appendix", "summary", "contents",
    "table of contents"
}

NON_PERSON_TERMS = {
    "report", "research", "assignment", "results", "introduction",
    "conclusion", "discussion", "services", "selection", "configuration",
    "installation", "implementation", "contents", "appendix", "appendixes",
    "references", "chapter", "table", "figure", "performance", "qualities",
    "cloud", "computing", "outlook", "calendar", "calander", "subject",
    "portfolio", "development", "professional", "personal"
}

INSTITUTION_PATTERNS = [
    r"\buniversity\b",
    r"\bcollege\b",
    r"\bfaculty\b",
    r"\bschool\b",
    r"\binstitute\b",
    r"\bdepartment\b",
    r"\bapplied sciences\b",
    r"\bhz university\b"
]

MONTH_WORDS = (
    "january|february|march|april|may|june|july|august|september|october|november|december|"
    "jan|feb|mar|apr|jun|jul|aug|sep|sept|oct|nov|dec|"
    "januari|februari|maart|april|mei|juni|juli|augustus|september|oktober|november|december"
)

DATE_PATTERN = (
    r"\b\d{1,2}(?:st|nd|rd|th)?[/-]\d{1,2}[/-]\d{2,4}\b|"
    r"\b\d{4}-\d{2}-\d{2}\b|"
    rf"\b\d{{1,2}}(?:st|nd|rd|th)?\s+(?:{MONTH_WORDS})\s+\d{{2,4}}\b|"
    rf"\b\d{{1,2}}(?:st|nd|rd|th)?\s+(?:{MONTH_WORDS})\b|"
    rf"\b(?:{MONTH_WORDS})\s+\d{{1,2}}(?:st|nd|rd|th)?,?\s+\d{{2,4}}\b|"
    rf"\b(?:{MONTH_WORDS})\s+\d{{4}}\b|"
    r"\b\d{4}\b"
)

DATE_CONTEXT_POSITIVE = [
    r"\bdate\b",
    r"\bpublication date\b",
    r"\bpublished\b",
    r"\breport date\b",
    r"\breleased\b",
    r"\bissued\b",
    r"\bversion\b",
    r"\bfinal report\b",
    r"\bsubmission date\b",
]

DATE_CONTEXT_NEGATIVE = [
    r"\breferences\b",
    r"\bbibliography\b",
    r"\bworks cited\b",
    r"\bet al\.\b",
    r"\bdoi\b",
    r"https?://",
    r"www\.",
    r"\bjournal\b",
    r"\bvol\.\b",
    r"\bno\.\b",
    r"\bpp\.\b",
]

AUTHOR_PREFIX_PATTERNS = [
    r"^(author|authors|written by|student|students|name|contributors?)\s*[:\-]?\s+",
    r"^by\s+"
]

PERSON_CONNECTORS = {"de", "van", "von", "der", "den", "ten", "ter", "la", "le", "du", "di", "op"}

YEAR_ONLY_RE = re.compile(r"^\d{4}$")


# Basic helpers

In [73]:
def safe_string(value: Any) -> str:
    return "" if value is None else str(value)

def normalize_whitespace(text: Any) -> str:
    return re.sub(r"\s+", " ", safe_string(text)).strip()

def dedupe_keep_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for item in items:
        clean = normalize_whitespace(item)
        key = clean.lower()
        if clean and key not in seen:
            seen.add(key)
            out.append(clean)
    return out

def load_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return [json_safe(v) for v in obj]

    # pandas / numpy scalars
    if hasattr(obj, "item"):
        try:
            return obj.item()
        except Exception:
            pass

    # datetime-like
    if isinstance(obj, datetime):
        return obj.isoformat()

    return obj


def write_json(path: Path, payload: Any) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(json_safe(payload), f, ensure_ascii=False, indent=2)

# Structural helpers

In [74]:
def is_page_number_line(line: str) -> bool:
    s = normalize_whitespace(line)
    return bool(re.fullmatch(r"\d+", s))

def is_table_of_contents_line(line: str) -> bool:
    s = normalize_whitespace(line)
    if "table of contents" in s.lower() or s.lower() == "contents":
        return True
    if re.search(r"\.{4,}\s*\d+$", s):
        return True
    if re.match(r"^\d+(\.\d+)*\s+.+\s+\d+$", s.lower()):
        return True
    return False

def looks_like_reference(sentence: str) -> bool:
    s = normalize_whitespace(sentence)
    if re.search(r"\bet al\.", s, flags=re.IGNORECASE):
        return True
    if re.search(r"\bdoi\b", s, flags=re.IGNORECASE):
        return True
    if re.search(r"https?://|www\.", s):
        return True
    if re.search(r"\breferences\b|\bbibliography\b", s, flags=re.IGNORECASE):
        return True
    if re.search(r"\(\s*[A-Z][A-Za-z].*\d{4}.*\)", s):
        return True
    return False

def is_short_heading_like(line: str) -> bool:
    s = normalize_whitespace(line)
    if not s:
        return False
    if len(s.split()) > 10:
        return False
    if s.lower() in SECTION_LABELS:
        return True
    if re.match(r"^\d+(\.\d+)*\s+[A-Za-z]", s):
        return True
    if s.istitle() and len(s.split()) <= 6:
        return True
    return False

def is_likely_org_line(line: str) -> bool:
    s = normalize_whitespace(line)
    return any(re.search(p, s, flags=re.IGNORECASE) for p in INSTITUTION_PATTERNS)

def is_structural_line(line: str) -> bool:
    s = normalize_whitespace(line)
    s_lower = s.lower()

    if not s:
        return True
    if is_page_number_line(s):
        return True
    if is_table_of_contents_line(s):
        return True
    if looks_like_reference(s):
        return True
    if is_short_heading_like(s):
        return True

    for prefix in STRUCTURAL_PREFIXES:
        if s_lower.startswith(prefix):
            return True

    return False

def looks_like_prose_line(line: str) -> bool:
    s = normalize_whitespace(line)
    if not s or is_structural_line(s):
        return False
    if len(s.split()) < 8:
        return False
    if re.search(r"\.{4,}", s):
        return False
    if sum(ch.isalpha() for ch in s) < 20:
        return False
    return True

# Front matter helpers

In [75]:
def extract_front_matter(text: str, max_chars: int = 15000) -> str:
    text = safe_string(text)
    if not text:
        return ""

    text = text.replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)

    stop_patterns = [
        r"\btable\s+of\s+contents\b",
        r"\bcontents\b",
        r"\binhoudsopgave\b",
        r"\babstract\b",
        r"\binleiding\b",
    ]

    earliest_match = None

    for pattern in stop_patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match and match.start() > 0:
            if earliest_match is None or match.start() < earliest_match:
                earliest_match = match.start()

    if earliest_match is not None:
        return text[:earliest_match].strip()

    return text[:max_chars].strip()

def get_front_lines_raw(text: str, max_chars: int = 15000, max_lines: int = 80) -> List[str]:
    front = extract_front_matter(text, max_chars=max_chars)
    return [line.strip() for line in front.splitlines() if line.strip()][:max_lines]

def split_joined_front_line(line: str) -> List[str]:
    s = normalize_whitespace(line)
    if not s:
        return []

    if any(re.search(p, s, flags=re.IGNORECASE) for p in INSTITUTION_PATTERNS):
        return [s]

    parts: List[str] = []

    author_match = re.search(
        r"([A-Z][a-z]+(?:\s+(?:de|den|der|van|von|ten|ter|op|la|le|du|di))?(?:\s+[A-Z][a-z]+)+"
        r"(?:\s*,\s*[A-Z][a-z]+(?:\s+(?:de|den|der|van|von|ten|ter|op|la|le|du|di))?(?:\s+[A-Z][a-z]+)+)*)$",
        s
    )

    author_part = ""
    if author_match:
        author_part = author_match.group(1).strip()
        s = s[:author_match.start()].strip()

    parts.append(s)

    final_parts = [normalize_whitespace(p) for p in parts if normalize_whitespace(p)]
    if author_part:
        final_parts.append(author_part)

    return final_parts

def get_front_lines(text: str, max_chars: int = 15000, max_lines: int = 80) -> List[str]:
    raw_lines = get_front_lines_raw(text, max_chars=max_chars, max_lines=max_lines)
    processed = []

    for line in raw_lines:
        processed.extend(split_joined_front_line(line))

    return [normalize_whitespace(x) for x in processed if normalize_whitespace(x)][:max_lines]

def get_title_page_lines(text: str, max_lines: int = 20) -> List[str]:
    text = safe_string(text).replace("\r", "\n")
    raw_lines = [line.strip() for line in text.splitlines() if line.strip()]

    stop_patterns = [
        r"^\s*contents\s*$",
        r"^\s*table\s+of\s+contents\s*$",
        r"^\s*inhoudsopgave\s*$",
        r"^\s*abstract\s*$",
        r"^\s*inleiding\s*$",
        r"^\s*\d+(\.\d+)*\s+[A-Za-z]",
    ]

    kept = []
    for line in raw_lines[:max_lines]:
        if any(re.search(p, line, flags=re.IGNORECASE) for p in stop_patterns):
            break
        kept.append(line)

    return kept

# Contributor helpers

In [76]:
def normalize_person_casing(name: str) -> str:
    parts = []
    for p in normalize_whitespace(name).split():
        if p.lower() in PERSON_CONNECTORS:
            parts.append(p.lower())
        elif p.isupper():
            parts.append(p.capitalize())
        else:
            parts.append(p)
    return " ".join(parts)

def is_valid_person_name(name: str) -> bool:
    name = normalize_whitespace(name)
    if len(name) < 5:
        return False

    parts = name.split()
    if len(parts) < 2 or len(parts) > 5:
        return False

    if any(ch.isdigit() for ch in name):
        return False
    if re.search(r"[•()\[\]{}:;\\/]", name):
        return False

    lowered_parts = [p.lower() for p in parts if p.lower() not in PERSON_CONNECTORS]
    if any(p in NON_PERSON_TERMS for p in lowered_parts):
        return False

    proper_tokens = 0
    for p in parts:
        if p.lower() in PERSON_CONNECTORS:
            continue
        if re.fullmatch(r"[A-Z][a-zà-ÿ'\-]{1,25}", p):
            proper_tokens += 1
        elif re.fullmatch(r"[A-Z]\.", p):  # allow initials
            proper_tokens += 1
        else:
            return False

    return proper_tokens >= 2

def extract_person_entities_from_row(row: pd.Series) -> List[str]:
    entities = row.get(ENTITIES_COLUMN, [])
    found = []

    if isinstance(entities, list):
        for ent in entities:
            if isinstance(ent, dict) and ent.get("label") == "PERSON":
                candidate = normalize_whitespace(ent.get("text", ""))
                if is_valid_person_name(candidate):
                    found.append(candidate)

    return dedupe_keep_order(found)

def clean_author_line_suffixes(line: str) -> str:
    s = normalize_whitespace(line)

    suffix_patterns = [
        r"\buitgave\s*:\s*#?\d+\b.*$",
        r"\bversion\s*:\s*[\w\.\-]+\b.*$",
        r"\bversie\s*:\s*[\w\.\-]+\b.*$",
        r"\breport\s*no\.?\s*[:\-]?\s*[\w\.\-]+\b.*$",
    ]

    for pattern in suffix_patterns:
        s = re.sub(pattern, "", s, flags=re.IGNORECASE).strip()

    return normalize_whitespace(s)

def extract_contributors_from_line(line: str) -> List[str]:
    s = normalize_whitespace(line)
    s = clean_author_line_suffixes(s)

    if not s:
        return []

    if any(re.search(p, s, flags=re.IGNORECASE) for p in INSTITUTION_PATTERNS):
        return []

    for pattern in AUTHOR_PREFIX_PATTERNS:
        s = re.sub(pattern, "", s, flags=re.IGNORECASE)

    parts = [normalize_whitespace(p) for p in re.split(r",| and | & |;", s) if normalize_whitespace(p)]
    normalized_parts = [normalize_person_casing(p) for p in parts]
    names = [p for p in normalized_parts if is_valid_person_name(p)]

    return dedupe_keep_order(names)

def extract_contributors(row: pd.Series, max_authors: int = 8) -> List[str]:
    text = safe_string(row.get(NLP_TEXT_COLUMN, "")) or safe_string(row.get(TEXT_COLUMN, ""))
    lines = get_title_page_lines(text, max_lines=20)

    ner_people = set(extract_person_entities_from_row(row))
    found = []

    for line in lines:
        if is_likely_org_line(line) or is_table_of_contents_line(line):
            continue

        names = extract_contributors_from_line(line)
        for n in names:
            found.append(n)

    for person in ner_people:
        if person not in found and any(person.lower() in line.lower() for line in lines):
            found.append(person)

    return dedupe_keep_order(found)[:max_authors]


# Title helpers

In [77]:
def is_date_like_line(line: str) -> bool:
    s = normalize_whitespace(line)
    if not s:
        return False
    if re.fullmatch(r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", s):
        return True
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
        return True
    return False

def is_title_noise(line: str) -> bool:
    s = normalize_whitespace(line)
    if not s:
        return True
    if s.lower() in SECTION_LABELS:
        return True
    if is_table_of_contents_line(s):
        return True
    if looks_like_reference(s):
        return True
    if is_date_like_line(s):
        return True
    return False

def extract_title_block(lines: List[str], max_lines_to_merge: int = 4) -> str:
    block = []

    for line in lines[:8]:
        s = normalize_whitespace(line)
        if not s:
            continue
        if is_table_of_contents_line(s):
            break
        if is_likely_org_line(s):
            break
        if re.search(DATE_PATTERN, s, flags=re.IGNORECASE):
            break
        if extract_contributors_from_line(s):
            break
        if s.lower() in SECTION_LABELS:
            break
        if re.fullmatch(r"\d+", s):
            continue

        block.append(s)
        if len(block) >= max_lines_to_merge:
            break

    return normalize_whitespace(" ".join(block))

def score_title_candidate(line: str, contributors: List[str]) -> float:
    s = normalize_whitespace(line)
    if is_title_noise(s):
        return -999.0
    if any(s.lower() == c.lower() for c in contributors):
        return -999.0

    wc = len(s.split())
    if wc < 2 or wc > 18:
        return -999.0

    score = 0.0
    if 2 <= wc <= 8:
        score += 2.0
    elif 9 <= wc <= 14:
        score += 1.0

    if is_likely_org_line(s):
        score -= 3.0
    if "," in s:
        score -= 2.0

    alpha = [ch for ch in s if ch.isalpha()]
    if alpha:
        uppercase_ratio = sum(ch.isupper() for ch in alpha) / len(alpha)
        if uppercase_ratio > 0.85:
            score += 1.2

    return score

def extract_title(row: pd.Series, contributors: Optional[List[str]] = None) -> str:
    contributors = contributors or []
    text = safe_string(row.get(NLP_TEXT_COLUMN, "")) or safe_string(row.get(TEXT_COLUMN, ""))
    lines = get_title_page_lines(text, max_lines=20)

    title_block = extract_title_block(lines)
    if len(title_block.split()) >= 2:
        return title_block

    candidates = []
    for line in lines:
        score = score_title_candidate(line, contributors)
        if score > -100:
            candidates.append((line, score))

    if not candidates:
        return ""

    candidates.sort(key=lambda x: x[1], reverse=True)
    best_title, best_score = candidates[0]
    return normalize_whitespace(best_title) if best_score >= 1.5 else ""


# Date helpers

In [78]:
def clean_date_text(date_text: str) -> str:
    date_text = normalize_whitespace(date_text).lower()
    date_text = re.sub(r"(\d{1,2})(st|nd|rd|th)\b", r"\1", date_text)
    return date_text.replace(",", "")

def extract_front_matter_for_dates(text: str, max_lines: int = 60) -> List[str]:
    text = safe_string(text).replace("\r", "\n")
    raw_lines = [line.strip() for line in text.splitlines() if line.strip()]

    stop_patterns = [
        r"^\s*abstract\s*$",
        r"^\s*table\s+of\s+contents\s*$",
        r"^\s*contents\s*$",
        r"^\s*inhoudsopgave\s*$",
        r"^\s*1(\.0+)?\s+(introduction|background|inleiding)\b",
        r"^\s*introduction\s*$",
        r"^\s*inleiding\s*$"
    ]

    kept = []
    for line in raw_lines[:max_lines]:
        if any(re.search(p, line, flags=re.IGNORECASE) for p in stop_patterns):
            break
        kept.append(line)

    return kept[:max_lines]

def line_looks_like_reference(line: str) -> bool:
    s = normalize_whitespace(line)
    if not s:
        return False
    if any(re.search(p, s, flags=re.IGNORECASE) for p in DATE_CONTEXT_NEGATIVE):
        return True
    if re.search(r"[A-Z][A-Za-z'\-]+,\s+[A-Z]\.", s):
        return True
    if re.search(r"\(\s*\d{4}[a-z]?\s*\)", s):
        return True
    return False

def score_date_candidate(date_text: str, line: str, line_index: int) -> float:
    s = normalize_whitespace(line)
    score = max(0, 10 - line_index) * 0.4

    for p in DATE_CONTEXT_POSITIVE:
        if re.search(p, s, flags=re.IGNORECASE):
            score += 2.0

    for p in DATE_CONTEXT_NEGATIVE:
        if re.search(p, s, flags=re.IGNORECASE):
            score -= 4.0

    num_dates = len(re.findall(DATE_PATTERN, s, flags=re.IGNORECASE))
    if num_dates > 2:
        score -= 2.0

    if len(s) < 40 and date_text in s:
        score += 1.5

    if YEAR_ONLY_RE.fullmatch(date_text):
        score -= 2.0

    return score

def parse_single_date(date_text: str, prefer_day_first: bool = True) -> Optional[datetime]:
    cleaned = clean_date_text(date_text)

    numeric_formats = (
        ["%d-%m-%Y", "%d/%m/%Y", "%d-%m-%y", "%d/%m/%y"]
        if prefer_day_first else
        ["%m-%d-%Y", "%m/%d/%Y", "%m-%d-%y", "%m/%d/%y"]
    )

    other_formats = [
        "%Y-%m-%d",
        "%d %B %Y", "%d %b %Y",
        "%d %B", "%d %b",
        "%B %d %Y", "%b %d %Y",
        "%B %Y", "%b %Y",
        "%Y"
    ]

    for fmt in numeric_formats + other_formats:
        try:
            dt = datetime.strptime(cleaned, fmt)
            if fmt in ("%B %Y", "%b %Y"):
                dt = dt.replace(day=1)
            if fmt == "%Y":
                dt = dt.replace(month=1, day=1)
            return dt
        except ValueError:
            continue
    return None

def parse_date_to_sortable(date_text: str, context_line: str = "") -> Optional[datetime]:
    dt_day_first = parse_single_date(date_text, prefer_day_first=True)
    dt_month_first = parse_single_date(date_text, prefer_day_first=False)

    if dt_day_first and not dt_month_first:
        return dt_day_first
    if dt_month_first and not dt_day_first:
        return dt_month_first
    return dt_day_first or dt_month_first

def extract_document_dates(text: str) -> Dict[str, Any]:
    lines = extract_front_matter_for_dates(text, max_lines=60)
    candidates = []

    for idx, line in enumerate(lines):
        s = normalize_whitespace(line)
        if not s or line_looks_like_reference(s):
            continue

        matches = re.findall(DATE_PATTERN, s, flags=re.IGNORECASE)
        for match in matches:
            date_text = normalize_whitespace(match)
            if not date_text:
                continue

            score = score_date_candidate(date_text, s, idx)
            candidates.append({
                "date": date_text,
                "line": s,
                "line_index": idx,
                "score": score
            })

    best_by_date = {}
    for item in candidates:
        key = item["date"].lower()
        if key not in best_by_date or item["score"] > best_by_date[key]["score"]:
            best_by_date[key] = item

    ranked = sorted(best_by_date.values(), key=lambda x: (-x["score"], x["line_index"]))
    valid = [x for x in ranked if x["score"] >= 1.0]

    parsed_valid = []
    for item in valid:
        parsed = parse_date_to_sortable(item["date"], context_line=item["line"])
        if parsed is not None:
            parsed_valid.append({**item, "parsed_date": parsed})

    parsed_valid = sorted(parsed_valid, key=lambda x: x["parsed_date"])

    if len(parsed_valid) >= 2:
        start_date = parsed_valid[0]["date"]
        end_date = parsed_valid[-1]["date"]
    elif len(parsed_valid) == 1:
        start_date = parsed_valid[0]["date"]
        end_date = ""
    else:
        start_date = ""
        end_date = ""

    return {
        "start_date": start_date,
        "end_date": end_date,
        "dates_found": [x["date"] for x in parsed_valid[:5]],
        "date_candidates": ranked[:10],
        "date_confidence": round(valid[0]["score"], 3) if valid else None
    }


# Description helpers

In [79]:
def extract_abstract_text(text: str, max_chars: int = 1600) -> str:
    text = safe_string(text)
    match = re.search(
        r"\babstract\b\s*(.*?)(?=\n\s*(table\s+of\s+contents|\d+(\.\d+)*\s+[A-Za-z]|introduction)\b|\Z)",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )
    if not match:
        return ""

    abstract = re.sub(r"\s+", " ", match.group(1)).strip()
    return abstract[:max_chars]

def extract_body_preview(text: str, max_chars: int = 500) -> str:
    lines = [line.strip() for line in safe_string(text).replace("\r", "\n").splitlines()]
    blocks = []
    current_block = []

    for line in lines:
        s = normalize_whitespace(line)

        if not s:
            if current_block:
                blocks.append(" ".join(current_block))
                current_block = []
            continue

        if is_structural_line(s):
            if current_block:
                blocks.append(" ".join(current_block))
                current_block = []
            continue

        if looks_like_prose_line(s):
            current_block.append(s)
        else:
            if current_block:
                blocks.append(" ".join(current_block))
                current_block = []

    if current_block:
        blocks.append(" ".join(current_block))

    for block in blocks:
        if len(block.split()) >= 30:
            return normalize_whitespace(block)[:max_chars]
    for block in blocks:
        if len(block.split()) >= 12:
            return normalize_whitespace(block)[:max_chars]

    return ""

# Classification

In [80]:
def build_research_group_text(title: str, description: str, cleaned_text: str, max_chars: int = 2500) -> str:
    pieces = []
    if title:
        pieces.append(f"Title: {title}")
    if description:
        pieces.append(f"Description: {description}")
    if cleaned_text:
        pieces.append(f"Key content: {cleaned_text[:1200]}")
    return normalize_whitespace(" ".join(pieces))[:max_chars]

def classify_research_group(
    title: str,
    description: str,
    cleaned_text: str,
    candidate_labels: List[str],
    top_k: int = 3
) -> Dict[str, Any]:
    classification_text = build_research_group_text(title, description, cleaned_text)

    if not classification_text.strip():
        return {
            "research_group": "",
            "research_group_confidence": None,
            "research_group_top3": []
        }

    result = zero_shot_classifier(
        classification_text,
        candidate_labels=candidate_labels,
        multi_label=False,
        hypothesis_template="Deze publicatie past het best binnen onderzoeksgroep {}."
    )

    labels = result["labels"]
    scores = result["scores"]

    top_items = [
        {"label": label, "score": round(float(score), 6)}
        for label, score in list(zip(labels, scores))[:top_k]
    ]

    best_label = labels[0] if labels else ""
    best_score = float(scores[0]) if scores else None

    if best_score is not None and best_score < RESEARCH_GROUP_CONFIDENCE_THRESHOLD:
        best_label = ""

    return {
        "research_group": best_label,
        "research_group_confidence": round(best_score, 6) if best_score is not None else None,
        "research_group_top3": top_items
    }

# Quality flags

In [81]:
def build_metadata_quality_flags(
    title: str,
    contributors: List[str],
    start_date: str,
    description: str,
    research_group: str
) -> List[str]:
    flags = []
    if not title:
        flags.append("missing_title")
    if not contributors:
        flags.append("missing_contributors")
    if not start_date:
        flags.append("missing_date")
    if not description:
        flags.append("missing_description")
    if not research_group:
        flags.append("missing_research_group")
    return flags

# Metadata pipeline

In [82]:
def extract_generic_metadata(row: pd.Series) -> Dict[str, Any]:
    document_id = safe_string(row.get(DOC_ID_COLUMN, ""))
    nlp_text = safe_string(row.get(NLP_TEXT_COLUMN, ""))
    cleaned_text = safe_string(row.get(TEXT_COLUMN, ""))
    raw_text = safe_string(row.get(RAW_TEXT_COLUMN, ""))

    working_text = nlp_text or cleaned_text
    contributors = extract_contributors(row)
    title = extract_title(row, contributors=contributors)
    date_info = extract_document_dates(working_text)

    abstract_text = extract_abstract_text(working_text)
    description = abstract_text or extract_body_preview(working_text)

    research_group_info = classify_research_group(
        title=title,
        description=description,
        cleaned_text=cleaned_text,
        candidate_labels=RESEARCH_GROUPS
    )

    quality_flags = build_metadata_quality_flags(
        title=title,
        contributors=contributors,
        start_date=date_info["start_date"],
        description=description,
        research_group=research_group_info["research_group"]
    )

    return {
        "id": document_id,
        "document_id": document_id,
        "source_file_name": row.get("source_file_name"),
        "source_file_path": row.get("source_file_path"),
        "pdf_metadata": row.get("pdf_metadata", {}),
        "page_count": row.get("page_count"),
        "bronze_extraction_status": row.get("bronze_extraction_status"),
        "used_ocr_fallback": row.get("used_ocr_fallback", False),
        "bronze_processed_at_utc": row.get("bronze_processed_at_utc"),
        "silver_processed_at_utc": row.get("silver_processed_at_utc"),
        "silver_nlp_processed_at_utc": row.get("silver_nlp_processed_at_utc"),
        "gold_processed_at_utc": datetime.now(timezone.utc).isoformat(),
        "title": title,
        "title_found": bool(title),
        "contributors": contributors,
        "contributors_found": bool(contributors),
        "start_date": date_info["start_date"],
        "end_date": date_info["end_date"],
        "dates_found": date_info["dates_found"],
        "date_confidence": date_info["date_confidence"],
        "description": description,
        "research_group": research_group_info["research_group"],
        "research_group_confidence": research_group_info["research_group_confidence"],
        "research_group_top3": research_group_info["research_group_top3"],
        "metadata_quality_flags": quality_flags,
        "metadata_debug": {
            "front_lines_sample": get_front_lines(working_text)[:12],
            "date_candidates": date_info["date_candidates"][:5]
        }
    }


# Batch run

In [83]:
silver_nlp_files = sorted(
    p for p in SILVER_FOLDER.glob("*_silver_nlp.json")
    if p.is_file()
)

if not silver_nlp_files:
    raise FileNotFoundError(f"No Silver NLP files found in {SILVER_FOLDER}")

results = []
manifest = []

for input_path in silver_nlp_files:
    payload = load_json(input_path)

    if isinstance(payload, list):
        if not payload:
            print(f"Skipping empty file: {input_path.name}")
            continue
        row_data = payload[0]
    elif isinstance(payload, dict):
        row_data = payload
    else:
        print(f"Skipping unsupported JSON format: {input_path.name}")
        continue

    df = pd.DataFrame([row_data])
    row = df.loc[0]

    result = extract_generic_metadata(row)
    output_path = GOLD_FOLDER / f"{result['document_id']}_gold_metadata.json"
    write_json(output_path, result)

    manifest_entry = {
        "document_id": result["document_id"],
        "source_file_name": result.get("source_file_name"),
        "input_json_path": str(input_path.resolve()),
        "output_json_path": str(output_path.resolve()),
        "title_found": result["title_found"],
        "contributors_found": result["contributors_found"],
        "start_date": result["start_date"],
        "research_group": result["research_group"],
        "research_group_confidence": result["research_group_confidence"],
        "metadata_quality_flags": result["metadata_quality_flags"],
        "gold_processed_at_utc": result["gold_processed_at_utc"],
    }

    manifest.append(manifest_entry)
    results.append(result)

    print(f"Saved Gold metadata: {output_path.name}")

Saved Gold metadata: doc_01_gold_metadata.json


# Reports

In [84]:
summary_df = pd.DataFrame([{
    "document_id": r["document_id"],
    "title_found": r["title_found"],
    "contributors_found": r["contributors_found"],
    "start_date": r["start_date"],
    "research_group": r["research_group"],
    "research_group_confidence": r["research_group_confidence"],
    "metadata_quality_flags": ", ".join(r["metadata_quality_flags"]) if r["metadata_quality_flags"] else "",
} for r in results])

print("\nGold metadata summary:")
print(summary_df)

validation_report = {
    "total_input_files": len(silver_nlp_files),
    "successful_documents": len(results),
    "documents_with_flags": sum(1 for r in results if r["metadata_quality_flags"]),
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "documents": manifest,
}

write_json(REPORTS_FOLDER / "gold_metadata_manifest.json", manifest)
write_json(REPORTS_FOLDER / "gold_metadata_validation_report.json", validation_report)

print(f"\nSaved manifest to: {REPORTS_FOLDER / 'gold_metadata_manifest.json'}")
print(f"Saved validation report to: {REPORTS_FOLDER / 'gold_metadata_validation_report.json'}")


Gold metadata summary:
  document_id  title_found  contributors_found     start_date research_group  \
0      doc_01         True                True  November 2018                  

   research_group_confidence  metadata_quality_flags  
0                   0.090765  missing_research_group  

Saved manifest to: ..\..\..\Data\gold\MBART\metadata\reports\gold_metadata_manifest.json
Saved validation report to: ..\..\..\Data\gold\MBART\metadata\reports\gold_metadata_validation_report.json
